# v10 · The piece `charset.py` names and the repo does not contain: a permutation index in [0, N!−1]
`v09/core/charset.py` says PRIVATE mode takes 'a permutation index (integer in [0, N!−1])'. No code in the repo converts between such an integer and a charset order. Below: Myrvold & Ruskey (2001) `rank1`/`unrank1`, written from the paper's pseudocode (read in full), checked against the paper's own n=4 table.

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


In [2]:
def unrank1(n, r, pi=None):
    pi = list(range(n)) if pi is None else pi
    while n > 0:
        pi[n-1], pi[r % n] = pi[r % n], pi[n-1]
        r //= n; n -= 1
    return pi
def rank1(pi):
    pi = list(pi); n = len(pi); inv = [0]*n
    for i, v in enumerate(pi): inv[v] = i
    r, mult = 0, 1
    for m in range(n, 1, -1):
        s = pi[m-1]
        pi[m-1], pi[inv[m-1]] = pi[inv[m-1]], pi[m-1]
        inv[s], inv[m-1] = inv[m-1], inv[s]
        r += s * mult; mult *= m
    return r
# the first eight rows of Fig. 1 of the paper (n = 4), transcribed
fig1 = {0: (1,2,3,0), 1: (3,2,0,1), 2: (1,3,0,2), 3: (1,2,0,3), 4: (2,3,1,0), 5: (2,0,3,1), 6: (3,0,1,2), 7: (2,0,1,3)}
assert all(tuple(unrank1(4, r)) == p for r, p in fig1.items())
assert sorted(rank1(unrank1(4, r)) for r in range(24)) == list(range(24))
print("n=4: unrank1 reproduces Fig. 1 rows 0-7; rank1∘unrank1 is a bijection on 0..23")

n=4: unrank1 reproduces Fig. 1 rows 0-7; rank1∘unrank1 is a bijection on 0..23


## At N = 97: the key is an integer of about 505 bits

In [3]:
import random
N = 97; F = math.factorial(N)
print("97! has", F.bit_length(), "bits")
rnd = random.Random(2026)
r = rnd.randrange(F)
pi = unrank1(N, r)
assert sorted(pi) == list(range(N)) and rank1(pi) == r
print("rank of a random charset order:", str(r)[:30] + "...", "(%d bits)" % r.bit_length())
# use it as a private charset with the v02 addressing class
hw2 = load("PtolemyDesktop/Callimachus/HyperWebster-Data-Storage/hyperwebster.py", "hw2")
pub = hw2.HyperWebster(); base = pub.characters
priv = hw2.HyperWebster("".join(base[i] for i in pi))
t = "The private order is a substitution"
print("public address :", pub.point_to_text(t))
print("private address:", priv.point_to_text(t))
assert priv.regenerate_text(priv.point_to_text(t)) == t

97! has 505 bits
rank of a random charset order: 440587842935793218461373961815... (504 bits)
public address : 2426202698443273741194746620062443157598793278889446799131611313540108
private address: 2608110931821973166458940924895133871064308018172022668847652761060194


Both directions are O(N) big-integer-free steps until the final integer. This is the missing conversion; `kcf.py` derives key bytes from a *hash* of the charset instead, so a charset order and a permutation index do not correspond in the current code.